# Materialize Graph inputs

Node properties use source column names for the standalone Graph demo. The ontology uses separate short aliases. Import mapping-manifest.json as a mapping guide, not an undocumented Fabric API definition.


In [ ]:
import sys
from pathlib import Path
BASE = Path('/lakehouse/default/Files/underwriting_demo')
sys.path.insert(0, str(BASE / 'src'))
from insurance_demo import build_data, validate, column_type, AS_OF, ROOT
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, BooleanType
TABLE_PREFIX = 'demo_'

def save_rows(name, rows):
    types = {'bit': BooleanType(), 'bigint': LongType(), 'float': DoubleType()}
    schema = StructType([StructField(field, types.get(column_type(field, rows), StringType()), True) for field in rows[0]])
    frame = spark.createDataFrame(rows, schema)
    for field in rows[0]:
        datatype = column_type(field, rows)
        if datatype == 'date':
            frame = frame.withColumn(field, F.to_date(field))
        elif datatype == 'datetime2(6)':
            frame = frame.withColumn(field, F.to_timestamp(field))
    destination = TABLE_PREFIX + name
    if spark.catalog.tableExists(destination):
        properties = {row.key: row.value for row in spark.sql(f'SHOW TBLPROPERTIES {destination}').collect()}
        assert properties.get('delta.columnMapping.mode', 'none') == 'none', 'Use a new managed demo table without column mapping'
    frame.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(destination)
    assert spark.table(destination).count() == len(rows)
    return frame

ENTITIES = {'Account': 'Accounts', 'Insured': 'Insureds', 'Broker': 'Brokers', 'Policy': 'Policies', 'Coverage': 'Coverages', 'Location': 'Locations', 'Exposure': 'Exposures', 'Asset': 'Assets', 'Claim': 'Claims', 'LossEvent': 'LossEvents', 'CATRisk': 'CATRisk', 'Peril': 'Perils', 'RiskEngineeringInspection': 'EngineeringInspections', 'EngineeringRecommendation': 'EngineeringRecommendations', 'Renewal': 'Renewals', 'RenewalSignal': 'RenewalSignals', 'Submission': 'Submissions', 'Industry': 'Industries', 'Country': 'Countries', 'MultinationalProgram': 'MultinationalPrograms', 'LocalPolicy': 'LocalPolicies', 'Underwriter': 'Underwriters', 'PricingScenario': 'PricingScenarios', 'UnderwritingDecision': 'UnderwritingDecisions', 'ProgramRequirement': 'ProgramRequirements'}
RELATIONS = [('Broker', 'SERVICES', 'Account', 'Accounts', 'BrokerId', 'AccountId'), ('Underwriter', 'OWNS', 'Account', 'Accounts', 'UnderwriterId', 'AccountId'), ('Account', 'CLASSIFIED_AS', 'Industry', 'Accounts', 'AccountId', 'IndustryId'), ('Account', 'HAS_INSURED', 'Insured', 'Insureds', 'AccountId', 'InsuredId'), ('Account', 'HOLDS', 'Policy', 'Policies', 'AccountId', 'PolicyId'), ('Policy', 'PROVIDES', 'Coverage', 'Coverages', 'PolicyId', 'CoverageId'), ('Account', 'OPERATES', 'Location', 'Locations', 'AccountId', 'LocationId'), ('Insured', 'OCCUPIES', 'Location', 'Locations', 'InsuredId', 'LocationId'), ('Location', 'IN_COUNTRY', 'Country', 'Locations', 'LocationId', 'CountryId'), ('Location', 'HAS_EXPOSURE', 'Exposure', 'Exposures', 'LocationId', 'ExposureId'), ('Location', 'CONTAINS', 'Asset', 'Assets', 'LocationId', 'AssetId'), ('Location', 'HAS_CAT', 'CATRisk', 'CATRisk', 'LocationId', 'CATId'), ('CATRisk', 'FOR_PERIL', 'Peril', 'CATRisk', 'CATId', 'PerilId'), ('Location', 'HAS_CLAIM', 'Claim', 'Claims', 'LocationId', 'ClaimId'), ('Claim', 'ON_POLICY', 'Policy', 'Claims', 'ClaimId', 'PolicyId'), ('Claim', 'ON_COVERAGE', 'Coverage', 'Claims', 'ClaimId', 'CoverageId'), ('Claim', 'FROM_EVENT', 'LossEvent', 'Claims', 'ClaimId', 'LossEventId'), ('LossEvent', 'OF_PERIL', 'Peril', 'LossEvents', 'LossEventId', 'PerilId'), ('Location', 'INSPECTED_BY', 'RiskEngineeringInspection', 'EngineeringInspections', 'LocationId', 'InspectionId'), ('RiskEngineeringInspection', 'ISSUED', 'EngineeringRecommendation', 'EngineeringRecommendations', 'InspectionId', 'RecommendationId'), ('Account', 'RENEWS', 'Renewal', 'Renewals', 'AccountId', 'RenewalId'), ('Renewal', 'HAS_SIGNAL', 'RenewalSignal', 'RenewalSignals', 'RenewalId', 'SignalId'), ('Account', 'HAS_SUBMISSION', 'Submission', 'Submissions', 'AccountId', 'SubmissionId'), ('Account', 'HAS_PROGRAM', 'MultinationalProgram', 'MultinationalPrograms', 'AccountId', 'ProgramId'), ('MultinationalProgram', 'HAS_LOCAL', 'LocalPolicy', 'LocalPolicies', 'ProgramId', 'LocalPolicyId'), ('LocalPolicy', 'IN_COUNTRY', 'Country', 'LocalPolicies', 'LocalPolicyId', 'CountryId'), ('MultinationalProgram', 'REQUIRES', 'ProgramRequirement', 'ProgramRequirements', 'ProgramId', 'RequirementId'), ('ProgramRequirement', 'MATCHES_LOCAL', 'LocalPolicy', 'ProgramRequirements', 'RequirementId', 'LocalPolicyId'), ('Renewal', 'CONSIDERS', 'PricingScenario', 'PricingScenarios', 'RenewalId', 'ScenarioId'), ('Renewal', 'HAS_DECISION', 'UnderwritingDecision', 'UnderwritingDecisions', 'RenewalId', 'DecisionId')]

for entity, table in ENTITIES.items():
    frame = spark.table(TABLE_PREFIX + table)
    if table in ('Exposures', 'CATRisk'):
        frame = frame.filter(F.col('SnapshotDate') == F.lit(AS_OF.isoformat()))
    frame.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('node_' + entity)
for source, relation, target, table, source_key, target_key in RELATIONS:
    frame = spark.table(TABLE_PREFIX + table)
    if table in ('Exposures', 'CATRisk'):
        frame = frame.filter(F.col('SnapshotDate') == F.lit(AS_OF.isoformat()))
    edges = frame.select(F.col(source_key).alias('SourceId'), F.col(target_key).alias('TargetId')).dropDuplicates()
    edges = edges.withColumn('EdgeId', F.concat_ws('_', F.lit(source + '_' + relation + '_' + target), 'SourceId', 'TargetId'))
    edges.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('edge_' + source + '_' + relation + '_' + target)
print('Build/refresh Graph and refresh ontology manually before rehearsal')
